<div style="background: linear-gradient(135deg, #0f172a 0%, #1e40af 100%); color: white; padding: 32px 40px; border-radius: 14px;"><div style="font-size: 0.85em; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.8;">Projet DataViz &mdash; Étape 3</div><div style="font-size: 1.9em; font-weight: 800; margin-top: 6px;">Analyse exploratoire (EDA)</div><div style="font-size: 1.05em; margin-top: 10px; opacity: 0.92;">Comprendre les indicateurs d’offre médicale en Île-de-France avant de construire le score de tension.</div></div>

## Ce qu’on cherche

1. **Qualité & granularité** des données — savoir ce qu’on manipule.
2. **APL généralistes** : comment l’accès au premier recours varie d’une commune à l’autre.
3. **Densité des spécialistes** : le contraste entre départements.
4. **Corrélations** entre indicateurs → est-ce qu’ils mesurent la même chose ou des choses différentes ? (décisif pour la **pondération du score**).
5. **Vue EPCI** : préparer le 2ᵉ niveau de zoom du dashboard.

> ⚠️ **Granularité asymétrique** : l’APL généralistes varie **à la commune** (1266 valeurs distinctes), alors que la densité des spécialistes est **au département** (8 valeurs, identiques pour toutes les communes d’un même département). On en tient compte dans chaque analyse.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"

df = pd.read_parquet("data/processed/communes_idf_consolide.parquet")
SPE = ["densite_cardio_100k", "densite_dermato_100k", "densite_ophtalmo_100k", "densite_gyneco_100k"]
LABELS = {"densite_cardio_100k": "Cardiologues", "densite_dermato_100k": "Dermatologues",
          "densite_ophtalmo_100k": "Ophtalmologues", "densite_gyneco_100k": "Gynécologues",
          "apl_generaliste": "APL généralistes"}
print(df.shape)
df.head()

## 1. Qualité & aperçu

In [ ]:
print("Communes :", len(df), "| Départements :", df.dep.nunique(), "| EPCI :", df.epci_siren.nunique())
print("Valeurs manquantes :", int(df.isna().sum().sum()))
df[["pop_commune", "apl_generaliste"] + SPE].describe().round(2)

## 2. APL généralistes — la variation à la commune

L’APL se lit en *nombre de consultations accessibles par habitant et par an*. Plus il est bas, plus l’accès au généraliste est tendu. On regarde la distribution sur les 1266 communes.

In [ ]:
apl = df["apl_generaliste"]
med = apl.median()
seuil = 2.5  # référence basse couramment utilisée pour les communes sous-dotées
print(f"Médiane IDF : {med:.2f} | moyenne : {apl.mean():.2f} | min {apl.min():.2f} / max {apl.max():.2f}")
sous = (apl < seuil)
print(f"Communes sous le seuil {seuil} : {sous.sum()} ({sous.mean()*100:.0f} %) "
      f"— soit {int(df.loc[sous, 'pop_commune'].sum()):,} habitants".replace(",", " "))

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(apl, bins=40, color="#1e40af", ax=ax)
ax.axvline(med, color="#0f172a", ls="--", lw=1.5, label=f"médiane {med:.2f}")
ax.axvline(seuil, color="#dc2626", ls="--", lw=1.5, label=f"seuil sous-dotation {seuil}")
ax.set(xlabel="APL généralistes (consultations / hab / an)", ylabel="Nombre de communes",
       title="Distribution de l’APL généralistes — 1266 communes d’Île-de-France")
ax.legend(); plt.tight_layout(); plt.show()

**Les communes les plus tendues** (APL le plus faible) — cibles prioritaires pour les élus :

In [ ]:
cols = ["nom_commune", "dep", "epci_nom", "pop_commune", "apl_generaliste"]
df.nsmallest(12, "apl_generaliste")[cols].reset_index(drop=True)

## 3. Densité des spécialistes — le contraste départemental

Les 4 spécialités étant au département, on travaille sur **8 valeurs**. On visualise l’écart entre départements, qui porte tout le récit du projet.

In [ ]:
dep_spe = df.groupby("dep")[SPE].first()
dep_spe.columns = [LABELS[c] for c in dep_spe.columns]

ax = dep_spe.plot(kind="bar", figsize=(11, 5), width=0.8,
                  color=["#1e40af", "#dc2626", "#0891b2", "#7c3aed"])
ax.set(xlabel="Département", ylabel="Densité / 100 000 hab.",
       title="Densité de spécialistes par département — Île-de-France")
plt.xticks(rotation=0); ax.legend(title=""); plt.tight_layout(); plt.show()

# écart max/min par spécialité
ecart = (dep_spe.max() / dep_spe.min()).round(1)
print("Écart densité max/min entre départements :")
print(ecart.to_string())

## 4. Corrélations entre indicateurs

Question clé pour le score : les indicateurs bougent-ils **ensemble** (redondants) ou capturent-ils des **dimensions différentes** (complémentaires) ? On agrège au **département** (APL pondéré par la population) pour comparer les 5 indicateurs sur une même base.

> Avec seulement 8 départements, ces corrélations sont **indicatives**, pas une preuve statistique.

In [ ]:
apl_dep = (df.assign(w=df.pop_commune)
             .groupby("dep")
             .apply(lambda g: np.average(g.apl_generaliste, weights=g.w), include_groups=False)
             .rename("apl_generaliste"))
mat = dep_spe.copy()
mat["APL généralistes"] = apl_dep.values

corr = mat.corr()
fig, ax = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Corrélations entre indicateurs (niveau département)")
plt.tight_layout(); plt.show()

## 5. Vue EPCI — préparer le 2ᵉ niveau de zoom

Le dashboard donne au président d’EPCI une vue de son territoire. On agrège les communes par EPCI : nombre de communes, population, et APL généralistes moyen (pondéré population).

In [ ]:
epci = (df.groupby(["epci_siren", "epci_nom", "epci_nature"])
          .apply(lambda g: pd.Series({
              "nb_communes": len(g),
              "population": g.pop_commune.sum(),
              "apl_gen_moy": np.average(g.apl_generaliste, weights=g.pop_commune),
              "apl_gen_min": g.apl_generaliste.min(),
          }), include_groups=False)
          .reset_index())
print(f"{len(epci)} EPCI couvrant l’Île-de-France")
epci.sort_values("apl_gen_moy").head(10).round(2)

## Synthèse — ce que l’EDA dit pour la suite

- **APL généralistes** : forte dispersion à la commune → c’est l’indicateur le plus fin, cœur de la vue maire.
- **Spécialistes** : écarts massifs entre départements (jusqu’à ×10+ sur la dermato) → l’indicateur qui justifie la mutualisation à l’échelle EPCI.
- **Corrélations** : à lire sur la heatmap — si les indicateurs sont faiblement corrélés, ils sont **complémentaires** et la pondération 1/3 chacun se défend ; s’ils sont très corrélés, on pourra simplifier ou repondérer.

**Étape suivante** → récupérer le KPI 3 (âge des médecins, data.drees) puis construire le **score de tension d’accès** par commune et par EPCI.